# 3-bosqich. Modellash va baholash

**Vazifa:** kelgusi 1–7 kun uchun kunlik **guruch sarfi** (`rice_kg`) va **tushumni** (`revenue_uzs`) bashorat qilish.

**Protokol**
| | |
|---|---|
| Featurelar | 2-bosqichda tanlangan 27 feature (`src/features.py`) |
| Target | `log(y)`, bashorat `exp()` bilan qaytariladi |
| Train | 2023-01-01 — 2024-12-31 (731 kun) |
| Giperparametr tanlash | faqat train ichida walk-forward CV (5 fold, har biri 73 kun) |
| Yakuniy test | 2025-01-01 — 2025-06-18 (169 kun), unga faqat bir marta tegiladi |
| Metrikalar | MAE, RMSE, MAPE, R², Bias (asl birliklarda) |

**Modellar** (oddiydan murakkabga):
1. *Baseline: o'tgan hafta.* `y(t) = y(t−7)`.
2. *Baseline: hafta kuni × oy.* Train'dagi o'rtacha.
3. *Ridge regressiya.* L2 regulyarizatsiyali chiziqli model, multikollinearlikka chidamli.
4. *Random Forest.*
5. *Gradient Boosting* (scikit-learn `HistGradientBoostingRegressor`).

Murakkab model faqat baseline'lardan sezilarli yaxshi bo'lsagina oqlanadi.

In [1]:
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import shap
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, cross_val_score

from src import style
from src.data import load_data, TARGETS, WEEKDAYS_UZ, MONTHS_UZ
from src.features import build_features, train_test_split_time, FEATURES
from src.models import (CV, SeasonalNaive, model_specs, metrics, oof_log_residuals, Forecaster)

warnings.filterwarnings("ignore")
style.setup_notebook()
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

df = load_data()
train, test = train_test_split_time(df)
X_tr, X_te = build_features(train), build_features(test)
TNAMES = {"rice_kg": "Guruch, kg", "revenue_uzs": "Tushum, so'm", "customers": "Mijozlar"}
print(f"X_train: {X_tr.shape}, X_test: {X_te.shape}")

X_train: (731, 27), X_test: (169, 27)


In [2]:
fig = go.Figure()
fig.add_scatter(x=train.date, y=train.rice_kg, mode="lines", line=dict(color=style.BLUE, width=1), name="Train (2023–2024)")
fig.add_scatter(x=test.date, y=test.rice_kg, mode="lines", line=dict(color=style.ORANGE, width=1), name="Test (2025)")
for k, (tr_idx, va_idx) in enumerate(CV.split(X_tr), start=1):
    d = train.iloc[va_idx]
    fig.add_vrect(x0=d.date.min(), x1=d.date.max(), fillcolor=style.GREY, opacity=0.12, line_width=0,
                  annotation_text=f"CV{k}", annotation_position="top left", annotation_font_size=10)
fig.update_layout(title="Ma'lumotni bo'lish: train, walk-forward CV validatsiya oynalari (kulrang) va test",
                  yaxis_title="Guruch, kg", height=360, legend=dict(orientation="h", y=1.1, x=1, xanchor="right"))
style.save(fig, "03_split")

## 1. Giperparametrlarni tanlash (walk-forward CV, faqat train)

Har bir CV foldda model faqat **o'tmish** ma'lumotida o'qitiladi va undan keyingi 73 kunda tekshiriladi.
Eng kichik fold ham 1 yillik ma'lumotda o'qitiladi, shuning uchun har bir oy kamida bir marta uchraydi.

In [3]:
fitted, cv_rows = {}, []
for t in TARGETS + ["customers"]:
    y_log = np.log(train[t])
    for name, (est, grid) in model_specs().items():
        gs = GridSearchCV(est, grid, cv=CV, scoring="neg_mean_absolute_error", n_jobs=-1).fit(X_tr, y_log)
        fitted[(t, name)] = gs.best_estimator_
        cv_rows.append({"target": TNAMES[t], "model": name, "CV MAE (log)": -gs.best_score_,
                        "≈ CV xato, %": (np.exp(-gs.best_score_) - 1) * 100,
                        "eng yaxshi parametrlar": gs.best_params_ or "—"})
cv_table = pd.DataFrame(cv_rows)
cv_table[cv_table.target != "Mijozlar"]

,target,model,CV MAE (log),"≈ CV xato, %",eng yaxshi parametrlar
0,"Guruch, kg",Baseline: hafta kuni × oy,0.195,21.529,—
1,"Guruch, kg",Ridge,0.082,8.507,{'ridge__alpha': 3}
2,"Guruch, kg",Random Forest,0.113,11.962,"{'max_features': 0.33, 'min_samples_leaf': 1}"
3,"Guruch, kg",Gradient Boosting,0.102,10.702,"{'l2_regularization': 1.0, 'learning_rate': 0...."
4,"Tushum, so'm",Baseline: hafta kuni × oy,0.203,22.556,—
5,"Tushum, so'm",Ridge,0.078,8.116,{'ridge__alpha': 3}
6,"Tushum, so'm",Random Forest,0.114,12.033,"{'max_features': 0.33, 'min_samples_leaf': 1}"
7,"Tushum, so'm",Gradient Boosting,0.096,10.089,"{'l2_regularization': 1.0, 'learning_rate': 0...."


## 2. Test natijalari (2025-yil, 169 kun)

In [4]:
rows, preds = [], {}
for t in TARGETS + ["customers"]:
    p_naive = SeasonalNaive().predict(df, t, test.index)
    preds[(t, "Baseline: o'tgan hafta")] = p_naive
    rows.append({"target": t, "model": "Baseline: o'tgan hafta", "to'plam": "test", **metrics(test[t], p_naive)})
    for name in model_specs():
        est = fitted[(t, name)]
        p_te, p_tr = np.exp(est.predict(X_te)), np.exp(est.predict(X_tr))
        preds[(t, name)] = p_te
        rows.append({"target": t, "model": name, "to'plam": "test", **metrics(test[t], p_te)})
        rows.append({"target": t, "model": name, "to'plam": "train", **metrics(train[t], p_tr)})
res = pd.DataFrame(rows)
MODELS = ["Baseline: o'tgan hafta", "Baseline: hafta kuni × oy", "Ridge", "Random Forest", "Gradient Boosting"]

def show(t):
    r = res[(res.target == t) & (res["to'plam"] == "test")].set_index("model").loc[MODELS]
    return r.drop(columns=["target", "to'plam"])

print("GURUCH (kg/kun)"); display(show("rice_kg").style.format("{:,.2f}").highlight_min(subset=["MAE", "RMSE", "MAPE, %"], color="#dbe8f5"))
print("TUSHUM (so'm/kun)"); display(show("revenue_uzs").style.format("{:,.2f}").highlight_min(subset=["MAE", "RMSE", "MAPE, %"], color="#dbe8f5"))

GURUCH (kg/kun)


,MAE,RMSE,"MAPE, %",R²,Bias
model,,,,,
Baseline: o'tgan hafta,12.39,16.44,27.71,-0.24,-0.60
Baseline: hafta kuni × oy,8.71,11.93,19.69,0.35,-1.24
Ridge,3.86,5.01,8.50,0.88,0.16
Random Forest,5.49,8.02,11.41,0.70,-1.39
Gradient Boosting,4.60,6.67,9.79,0.80,0.14


TUSHUM (so'm/kun)


,MAE,RMSE,"MAPE, %",R²,Bias
model,,,,,
Baseline: o'tgan hafta,"3,226,706.46","4,264,591.66",28.67,-0.30,"-179,848.51"
Baseline: hafta kuni × oy,"2,271,426.78","3,108,008.74",20.06,0.31,"-454,478.50"
Ridge,"893,217.12","1,159,348.30",7.62,0.90,"-72,959.12"
Random Forest,"1,338,116.44","2,016,042.54",10.74,0.71,"-440,897.47"
Gradient Boosting,"1,080,124.57","1,608,549.78",8.97,0.81,"-36,965.33"


In [5]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["Guruch: test MAPE, %", "Tushum: test MAPE, %"], shared_yaxes=True)
for j, t in enumerate(TARGETS, start=1):
    r = show(t)
    colors = [style.GREY, style.GREY, style.BLUE, "#7a9cc6", "#7a9cc6"]
    fig.add_bar(y=r.index, x=r["MAPE, %"], orientation="h", marker_color=colors, showlegend=False,
                text=[f"{v:.1f}%" for v in r["MAPE, %"]], textposition="outside", row=1, col=j)
    fig.update_xaxes(range=[0, r["MAPE, %"].max() * 1.25], row=1, col=j)
fig.update_yaxes(autorange="reversed")
fig.update_layout(title="Modellarni solishtirish: o'rtacha foizli xato (test, kichikroq — yaxshiroq)", height=360)
style.save(fig, "03_model_comparison")

### 2.1. Overfitting tekshiruvi: train va test xatosi

In [6]:
ov = res[res.model.isin(MODELS[1:])].pivot_table(index=["target", "model"], columns="to'plam", values="MAPE, %")
ov["farq (test − train), p.p."] = ov["test"] - ov["train"]
ov.loc[TARGETS].rename(index=TNAMES, level=0)

to'plam                                  test  train  \
target       model                                     
Guruch, kg   Baseline: hafta kuni × oy 19.693 16.342   
             Gradient Boosting          9.792  7.143   
             Random Forest             11.411  3.987   
             Ridge                      8.504  7.614   
Tushum, so'm Baseline: hafta kuni × oy 20.058 17.062   
             Gradient Boosting          8.969  6.745   
             Random Forest             10.744  3.961   
             Ridge                      7.615  7.188   

to'plam                                 farq (test − train), p.p.  
target       model                                                 
Guruch, kg   Baseline: hafta kuni × oy                      3.352  
             Gradient Boosting                              2.649  
             Random Forest                                  7.424  
             Ridge                                          0.890  
Tushum, so'm Baseline: hafta kuni × oy                      2.996  
             Gradient Boosting                              2.225  
             Random Forest                                  6.783  
             Ridge                                          0.427

**Kuzatuvlar.**
- Ikkala baseline juda zaif: "o'tgan hafta" xatosi ~28%, "hafta kuni × oy" xatosi ~20%.
  Kalendar talabning faqat bir qismini tushuntiradi. Ob-havo va tadbirlar ham zarur.
- **Ridge eng yaxshi natija berdi:** guruch uchun MAPE ≈ 8.5%, tushum uchun ≈ 7.6%. Uning train va test xatosi deyarli bir xil, overfitting yo'q.
- **Random Forest** train'da 4% xato, test'da ~11%: aniq overfitting.
  **Gradient Boosting** undan yaxshiroq, lekin Ridge'dan ortda qoladi.
- Nega oddiy chiziqli model yutdi? 2-bosqichda ko'rsatilgandek, `log(y)` da omillar ta'siri **qo'shiluvchi** va interaksiyalar yo'q.
  Daraxtlar bunday tuzilishni ko'p bo'linishlar orqali taxminan qurishga majbur, 731 kunlik ma'lumot esa buning uchun kam.
  Ma'lumot tuzilishiga mos keladigan oddiy model murakkab modeldan yaxshiroq ishladi.

## 3. Ikki yondashuv: to'g'ridan-to'g'ri yoki mijozlar orqali

1-bosqichda guruch va tushum mijozlar soniga deyarli chiziqli bog'liq ekani ko'rindi.
Muqobil yondashuv: avval mijozlar sonini bashorat qilish, keyin uni train'dagi o'rtacha nisbatga ko'paytirish
(guruch/mijoz va tushum/mijoz).

In [7]:
ratio = {t: train[t].sum() / train["customers"].sum() for t in TARGETS}
rows = []
for t in TARGETS:
    p_direct = preds[(t, "Ridge")]
    p_via = preds[("customers", "Ridge")] * ratio[t]
    rows.append({"target": TNAMES[t], "yondashuv": "To'g'ridan-to'g'ri (Ridge)", **metrics(test[t], p_direct)})
    rows.append({"target": TNAMES[t], "yondashuv": f"Mijozlar (Ridge) × {ratio[t]:,.3f}", **metrics(test[t], p_via)})
pd.DataFrame(rows).set_index(["target", "yondashuv"])

MAE          RMSE  MAPE, %  \
target       yondashuv                                                          
Guruch, kg   To'g'ridan-to'g'ri (Ridge)          3.862         5.011    8.504   
             Mijozlar (Ridge) × 0.220            3.851         4.979    8.493   
Tushum, so'm To'g'ridan-to'g'ri (Ridge)    893,217.123 1,159,348.295    7.615   
             Mijozlar (Ridge) × 55,971.307 893,221.407 1,147,476.820    7.763   

                                              R²        Bias  
target       yondashuv                                        
Guruch, kg   To'g'ridan-to'g'ri (Ridge)    0.885       0.161  
             Mijozlar (Ridge) × 0.220      0.886       0.181  
Tushum, so'm To'g'ridan-to'g'ri (Ridge)    0.904 -72,959.117  
             Mijozlar (Ridge) × 55,971.307 0.906 -60,217.650

In [8]:
# Mijozlar sonini mukammal bilsak ham, nisbat tebranishi tufayli qoladigan xato (pastki chegara)
floor = {TNAMES[t]: metrics(test[t], test["customers"] * ratio[t])["MAPE, %"] for t in TARGETS}
pd.Series(floor, name="MAPE, % (mijozlar soni aniq ma'lum bo'lganda)").to_frame()

,"MAPE, % (mijozlar soni aniq ma'lum bo'lganda)"
"Guruch, kg",4.468
"Tushum, so'm",3.472


**Xulosa.** Ikkala yondashuv deyarli bir xil natija beradi. To'g'ridan-to'g'ri model qoldiriladi, chunki u soddaroq
(bitta model) va har bir target o'z omillarini o'zi o'rganadi. Masalan, maosh haftasida tushum guruchdan ko'proq oshadi.

Muhim kuzatuv: mijozlar soni **aniq** ma'lum bo'lsa ham, guruch xatosi ~4.5% atrofida qoladi, chunki bir mijozning
guruch iste'moli kundan kunga tebranadi. Ya'ni ~8.5% xatoning yarmiga yaqini hech qanday model kamaytira olmaydigan tasodifiy shovqin.

## 4. Haroratni modelda qoldirish kerakmi? (CV bilan tekshirish)

2-bosqichda haroratning sof ta'siri chegaraviy ahamiyatli chiqqan edi. Qarorni train ichidagi CV bilan tekshiramiz.

In [9]:
rows = []
for t in TARGETS:
    est = fitted[(t, "Ridge")]
    y_log = np.log(train[t])
    for label, cols in [("temp_c bilan", FEATURES), ("temp_c siz", [c for c in FEATURES if c != "temp_c"])]:
        s = -cross_val_score(clone(est), X_tr[cols], y_log, cv=CV, scoring="neg_mean_absolute_error")
        rows.append({"target": TNAMES[t], "variant": label, "CV MAE (log), o'rtacha": s.mean(), "std": s.std()})
pd.DataFrame(rows).set_index(["target", "variant"])

CV MAE (log), o'rtacha   std
target       variant                                   
Guruch, kg   temp_c bilan                   0.082 0.008
             temp_c siz                     0.083 0.009
Tushum, so'm temp_c bilan                   0.078 0.004
             temp_c siz                     0.079 0.005

**Xulosa.** Harorat bilan va haroratsiz variantlar orasidagi farq CV shovqini darajasida.
Harorat modelda qoldiriladi: u zarar keltirmaydi, mantiqan asoslangan va mavsumdan tashqari issiq yoki sovuq kunlarni ushlaydi.

## 5. Eng yaxshi model (Ridge): bashorat va haqiqiy qiymatlar

In [10]:
# Bashorat oralig'i: walk-forward CV dagi out-of-fold qoldiqlarning 10% va 90% kvantillari (split-conformal)
best = {t: fitted[(t, "Ridge")] for t in TARGETS}
resid_q = {}
for t in TARGETS:
    r = oof_log_residuals(best[t], X_tr, np.log(train[t]))
    resid_q[t] = {0.1: np.quantile(r, 0.1), 0.9: np.quantile(r, 0.9)}

band = {}
for t in TARGETS:
    p = best[t].predict(X_te)
    band[t] = pd.DataFrame({"P50": np.exp(p), "P10": np.exp(p + resid_q[t][0.1]), "P90": np.exp(p + resid_q[t][0.9])},
                           index=test.index)

cov = {TNAMES[t]: {"P10–P90 oralig'iga tushgan kunlar, %": ((test[t] >= band[t].P10) & (test[t] <= band[t].P90)).mean() * 100,
                   "P90 dan oshgan kunlar, %": (test[t] > band[t].P90).mean() * 100,
                   "Oraliqning o'rtacha kengligi": (band[t].P90 - band[t].P10).mean()} for t in TARGETS}
pd.DataFrame(cov).T

,"P10–P90 oralig'iga tushgan kunlar, %","P90 dan oshgan kunlar, %",Oraliqning o'rtacha kengligi
"Guruch, kg",80.473,7.692,12.731
"Tushum, so'm",84.024,8.876,"3,008,576.994"


In [11]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=["Guruch sarfi, kg/kun", "Tushum, mln so'm/kun"])
for i, (t, scale, color) in enumerate([("rice_kg", 1, style.BLUE), ("revenue_uzs", 1e6, style.ORANGE)], start=1):
    b = band[t] / scale
    fig.add_scatter(x=test.date, y=b.P90, mode="lines", line=dict(width=0), showlegend=False, hoverinfo="skip", row=i, col=1)
    fig.add_scatter(x=test.date, y=b.P10, mode="lines", line=dict(width=0), fill="tonexty",
                    fillcolor="rgba(140,140,140,0.25)", name="P10–P90 oralig'i", showlegend=(i == 1), row=i, col=1)
    fig.add_scatter(x=test.date, y=test[t] / scale, mode="lines+markers", marker=dict(size=3), line=dict(color="#222", width=1),
                    name="Haqiqiy", showlegend=(i == 1), row=i, col=1)
    fig.add_scatter(x=test.date, y=b.P50, mode="lines", line=dict(color=color, width=2), name="Bashorat (Ridge)",
                    showlegend=(i == 1), row=i, col=1)
fig.update_layout(title="Test davri (2025): haqiqiy va bashorat qilingan qiymatlar", height=640,
                  legend=dict(orientation="h", y=1.07, x=1, xanchor="right"))
style.save(fig, "03_test_forecast")

In [12]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["Guruch, kg", "Tushum, mln so'm"])
for j, (t, scale, color) in enumerate([("rice_kg", 1, style.BLUE), ("revenue_uzs", 1e6, style.ORANGE)], start=1):
    yt, yp = test[t] / scale, band[t].P50 / scale
    lo, hi = min(yt.min(), yp.min()), max(yt.max(), yp.max())
    fig.add_scatter(x=[lo, hi], y=[lo, hi], mode="lines", line=dict(color=style.GREY, dash="dash"), showlegend=False, row=1, col=j)
    fig.add_scatter(x=yp, y=yt, mode="markers", marker=dict(color=color, size=6, opacity=0.6), showlegend=False,
                    text=test.date.dt.strftime("%Y-%m-%d"), hovertemplate="%{text}<br>bashorat %{x:.1f}<br>haqiqiy %{y:.1f}",
                    row=1, col=j)
    m = metrics(test[t], band[t].P50)
    fig.layout.annotations[j - 1].text += f"   (R² = {m['R²']:.3f}, MAPE = {m['MAPE, %']:.1f}%)"
fig.update_xaxes(title_text="Bashorat")
fig.update_yaxes(title_text="Haqiqiy")
fig.update_layout(title="Haqiqiy va bashorat (test); punktir — mukammal bashorat chizig'i", height=420)
style.save(fig, "03_scatter")

## 6. Xatolar tahlili: model qayerda adashadi?

In [13]:
err = test[["date", "weekday", "month", "rice_kg", "revenue_uzs", "is_holiday", "eid", "ramadan", "promotion",
            "local_event", "football_match", "snow", "rain"]].copy()
err["bashorat"] = band["rice_kg"].P50
err["xato, %"] = (err["bashorat"] - err["rice_kg"]) / err["rice_kg"] * 100
err["|xato|, %"] = err["xato, %"].abs()

fig = go.Figure()
for wd in range(7):
    fig.add_box(y=err.loc[err.weekday == wd, "xato, %"], name=WEEKDAYS_UZ[wd], marker_color=style.BLUE, boxmean=True,
                showlegend=False)
fig.add_hline(y=0, line=dict(color="#444", width=1))
fig.update_layout(title="Guruch bashorati xatosi hafta kunlari bo'yicha (test; musbat — ortiqcha bashorat)",
                  yaxis_title="Xato, %", height=380)
style.save(fig, "03_error_by_weekday")

In [14]:
flags = {"is_holiday": "Bayram", "eid": "Hayit", "ramadan": "Ramazon", "promotion": "Aksiya",
         "local_event": "Mahalliy tadbir", "football_match": "Futbol o'yini", "snow": "Qor", "rain": "Yomg'ir"}
rows = [{"kun turi": "Oddiy kun (hech bir belgi yo'q)", "n": int((~err[list(flags)].any(axis=1)).sum()),
         "MAPE, %": err.loc[~err[list(flags)].any(axis=1), "|xato|, %"].mean(),
         "o'rtacha xato (bias), %": err.loc[~err[list(flags)].any(axis=1), "xato, %"].mean()}]
for c, nm in flags.items():
    m = err[c]
    rows.append({"kun turi": nm, "n": int(m.sum()), "MAPE, %": err.loc[m, "|xato|, %"].mean(),
                 "o'rtacha xato (bias), %": err.loc[m, "xato, %"].mean()})
pd.DataFrame(rows)

,kun turi,n,"MAPE, %","o'rtacha xato (bias), %"
0,Oddiy kun (hech bir belgi yo'q),86,9.253,1.556
1,Bayram,4,10.798,5.475
2,Hayit,2,4.077,-4.077
3,Ramazon,30,7.688,-1.159
4,Aksiya,14,11.561,3.990
5,Mahalliy tadbir,9,6.581,-0.959
6,Futbol o'yini,9,7.026,-0.633
7,Qor,9,13.182,10.536
8,Yomg'ir,38,7.443,0.430


In [15]:
top = err.sort_values("|xato|, %", ascending=False).head(10).copy()
top["belgilar"] = top[list(flags)].apply(lambda r: ", ".join(flags[c] for c in flags if r[c]) or "—", axis=1)
top["kun"] = top.weekday.map(lambda i: WEEKDAYS_UZ[i])
top[["date", "kun", "rice_kg", "bashorat", "xato, %", "belgilar"]].rename(columns={"rice_kg": "haqiqiy, kg", "bashorat": "bashorat, kg"})

,date,kun,"haqiqiy, kg","bashorat, kg","xato, %",belgilar
760,2025-01-30,Pay,22.200,31.327,41.111,"Aksiya, Qor"
776,2025-02-15,Shan,39.200,50.746,29.455,—
868,2025-05-18,Yak,50.000,64.579,29.159,—
740,2025-01-10,Juma,32.700,41.896,28.121,—
895,2025-06-14,Shan,56.300,71.403,26.826,—
769,2025-02-08,Shan,40.700,50.861,24.965,—
762,2025-02-01,Shan,27.700,34.369,24.076,Qor
842,2025-04-22,Sesh,42.000,51.038,21.519,—
898,2025-06-17,Sesh,37.700,45.604,20.965,—
862,2025-05-12,Dush,31.700,38.292,20.796,—


**Kuzatuvlar.**
- **Shanba kunlari** 2025-yilda model talabni o'rtacha ~7% ortiqcha bashorat qiladi. Eng katta 10 xatoning 4 tasi ham Shanbaga to'g'ri keladi.
  Ehtimol, 2025-yilda Shanba kunlari 2023–2024 dagidek gavjum emas. Bu **o'zgarishni kuzatish** (monitoring) kerakligini ko'rsatadi.
  Yakuniy model 2025 ma'lumotlarini ham o'z ichiga oladi, shuning uchun bu siljish qisman hisobga olinadi.
- **Qorli kunlar** eng qiyin: MAPE ~13%, model talabni ~10% ortiqcha bashorat qiladi. 2025-yilgi qorli kunlarda
  talab train'dagidan ham ko'proq tushgan.
- Hayit (2 kun) va bayram (4 kun) uchun test'da ham namuna juda kichik, shuning uchun ular bo'yicha xulosa chiqarib bo'lmaydi.
- Eng katta xatolarning deyarli barchasi **ortiqcha bashorat**. Guruch rejalashtirishda bu xavfsizroq yo'nalish:
  yetishmovchilik emas, ortiqcha qoladi.

## 7. Interpretatsiya: model qaysi omillarga tayanadi?

Ridge modeli uchun har bir featurening test kunlaridagi hissasi (SHAP, chiziqli model uchun aniq hisoblanadi)
Gradient Boosting'ning SHAP qiymatlari bilan solishtiriladi. Dummy'lar o'z guruhiga yig'iladi:
6 ta `wd_*` → "Hafta kuni", 11 ta `m_*` → "Oy".

In [16]:
GROUP = {"temp_c": "Harorat", "rain": "Yomg'ir", "snow": "Qor", "is_holiday": "Bayram", "ramadan": "Ramazon",
         "eid": "Hayit", "salary_week": "Maosh haftasi", "promotion": "Aksiya", "local_event": "Mahalliy tadbir",
         "football_match": "Futbol o'yini", **{f"wd_{i}": "Hafta kuni" for i in range(1, 7)},
         **{f"m_{i}": "Oy" for i in range(2, 13)}}

def grouped_importance(sv):
    s = pd.DataFrame(sv, columns=FEATURES).T.groupby(GROUP).sum().T   # bir guruh dummy'lari hissasi qo'shiladi
    return s.abs().mean().sort_values()

t = "rice_kg"
ridge = best[t]
bg = ridge[0].transform(X_tr)
lin = shap.LinearExplainer(ridge[-1], shap.maskers.Independent(bg, max_samples=len(bg)))
sv_ridge = lin.shap_values(ridge[0].transform(X_te))
gb = fitted[(t, "Gradient Boosting")]
sv_gb = shap.TreeExplainer(gb).shap_values(X_te)

imp = pd.DataFrame({"Ridge": grouped_importance(sv_ridge), "Gradient Boosting": grouped_importance(sv_gb)})
imp = imp.sort_values("Ridge")
fig = go.Figure()
fig.add_bar(y=imp.index, x=imp["Ridge"], orientation="h", name="Ridge", marker_color=style.BLUE)
fig.add_bar(y=imp.index, x=imp["Gradient Boosting"], orientation="h", name="Gradient Boosting", marker_color="#a9c1dd")
fig.update_layout(barmode="group", title="Omillarning ahamiyati: o'rtacha |SHAP| (log guruch, test)",
                  xaxis_title="o'rtacha |SHAP| (log shkala)", height=480, legend=dict(orientation="h", y=1.08, x=1, xanchor="right"))
style.save(fig, "03_shap")

**Xulosa.**
- Ikkala modelda ham **hafta kuni** eng muhim omil. Undan keyin Ramazon, maosh haftasi, yomg'ir, qor va aksiya keladi.
  Bu tartib 2-bosqichdagi statistik tahlil bilan mos keladi.
- **Mavsum ta'siri modellarda turlicha taqsimlangan.** Ridge uni asosan **oy**ga beradi, Gradient Boosting esa **harorat**ga.
  Bu 2-bosqichda aniqlangan multikollinearlikning amaldagi ko'rinishi: harorat dispersiyasining 93% i oy bilan tushuntiriladi,
  shuning uchun modellar bir xil "mavsum" signalini ikki xil feature orqali olishi mumkin.
  Oy va haroratning **birgalikdagi** ahamiyati ikkala modelda deyarli bir xil.
- Gradient Boosting **Hayit** va **bayramni** umuman ishlatmaydi (ahamiyati 0): daraxtlar kam uchraydigan belgilarda bo'linish qilmaydi.
  Ridge esa ularning ta'sirini koeffitsient orqali saqlaydi. Bu Ridge ustunligining yana bir sababi.

## 8. Yakuniy model va kelgusi 7 kun uchun bashorat

Yakuniy model **barcha 900 kunda** (train + test) qayta o'qitiladi. Giperparametrlar CV da tanlanganlari qoladi,
bashorat oralig'i esa yangi walk-forward qoldiqlaridan hisoblanadi.
Model `models/forecaster.joblib` ga saqlanadi va `Forecaster.load().predict(...)` orqali ishlatiladi.

**Ufq haqida:** modelda lag featurelar yo'q, shuning uchun uning aniqligi 1 kunlik va 7 kunlik ufqda bir xil.
Real hayotda esa ob-havo prognozining xatosi ufq uzaygan sari oshadi.

In [17]:
X_all = build_features(df)
final = Forecaster()
for t in TARGETS:
    y_log = np.log(df[t])
    final.estimators[t] = clone(best[t]).fit(X_all, y_log)
    r = oof_log_residuals(best[t], X_all, y_log)
    final.resid_q[t] = {0.1: np.quantile(r, 0.1), 0.9: np.quantile(r, 0.9)}
path = final.save()
print("Saqlandi:", path.relative_to(path.parents[1]))

Saqlandi: models/forecaster.joblib


In [18]:
# Ssenariy: 2025-06-19 … 2025-06-25. Harorat — 2023–2024 yillarning shu kunlaridagi o'rtacha (±3 kun),
# yog'ingarchilik yo'q, aksiya va tadbirlar rejalashtirilmagan. Real ishlatishda bu qiymatlar ob-havo prognozi va
# tadbirlar rejasidan olinadi.
future = pd.DataFrame({"date": pd.date_range("2025-06-19", periods=7, freq="D")})
doy = df.date.dt.dayofyear
future["temp_c"] = [df.loc[(df.year < 2025) & ((doy - d.dayofyear).abs() <= 3), "temp_c"].mean() for d in future.date]

fc = Forecaster.load().predict(future)
show_fc = pd.DataFrame({
    "sana": fc.date.dt.strftime("%Y-%m-%d"),
    "kun": fc.date.dt.weekday.map(lambda i: WEEKDAYS_UZ[i]),
    "harorat, °C": future.temp_c.round(1),
    "guruch P50, kg": fc.rice_kg_P50, "guruch P10–P90, kg": [f"{a:.1f} – {b:.1f}" for a, b in zip(fc.rice_kg_P10, fc.rice_kg_P90)],
    "tushum P50, mln": fc.revenue_uzs_P50 / 1e6,
    "tushum P10–P90, mln": [f"{a / 1e6:.1f} – {b / 1e6:.1f}" for a, b in zip(fc.revenue_uzs_P10, fc.revenue_uzs_P90)],
})
show_fc

,sana,kun,"harorat, °C","guruch P50, kg","guruch P10–P90, kg","tushum P50, mln","tushum P10–P90, mln"
0,2025-06-19,Pay,34.200,60.442,52.7 – 68.6,15.645,14.0 – 17.8
1,2025-06-20,Juma,34.200,74.759,65.2 – 84.9,19.177,17.1 – 21.8
2,2025-06-21,Shan,34.400,84.255,73.5 – 95.7,21.583,19.3 – 24.5
3,2025-06-22,Yak,34.300,70.067,61.1 – 79.6,18.094,16.1 – 20.6
4,2025-06-23,Dush,34.000,42.650,37.2 – 48.4,10.629,9.5 – 12.1
5,2025-06-24,Sesh,34.300,45.348,39.6 – 51.5,11.197,10.0 – 12.7
6,2025-06-25,Chor,34.200,47.659,41.6 – 54.1,11.838,10.6 – 13.4


In [19]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["Guruch, kg", "Tushum, mln so'm"])
x = [f"{d:%d-%b}<br>{WEEKDAYS_UZ[d.weekday()]}" for d in fc.date]
for j, (t, scale, color) in enumerate([("rice_kg", 1, style.BLUE), ("revenue_uzs", 1e6, style.ORANGE)], start=1):
    p50, p10, p90 = fc[f"{t}_P50"] / scale, fc[f"{t}_P10"] / scale, fc[f"{t}_P90"] / scale
    fig.add_bar(x=x, y=p50, marker_color=color, showlegend=False,
                error_y=dict(type="data", symmetric=False, array=p90 - p50, arrayminus=p50 - p10, color="#444", thickness=1.2),
                row=1, col=j)
fig.update_layout(title="Kelgusi 7 kun bashorati (ustun — P50, chiziq — P10–P90; ssenariy: yog'ingarchilik va aksiya yo'q)",
                  height=400)
style.save(fig, "03_next_week")

**Amaliy tavsiya.** Guruch zaxirasini **P90** bo'yicha rejalashtirish mumkin: kunlarning ~90% ida ehtiyoj shu miqdordan oshmaydi.
P50 bo'yicha rejalashtirilsa, kunlarning yarmida yetishmovchilik bo'ladi.
Qaysi kvantil tanlanishi ortiqcha guruch va yetishmovchilik xarajatlarining nisbatiga bog'liq.

## 9. Yakuniy xulosa

| | Guruch | Tushum |
|---|---|---|
| Eng yaxshi model | Ridge (log y) | Ridge (log y) |
| Test MAPE | ~8.5% | ~7.6% |
| Test R² | ~0.89 | ~0.90 |
| Eng yaxshi baseline MAPE | ~20% | ~20% |

1. Ridge eng yaxshi baseline'dan **2.3–2.6 baravar** aniqroq, murakkab daraxtli modellardan ham yaxshiroq ishladi.
   Buning sababi ma'lumot tuzilishi: log shkalada ta'sirlar qo'shiluvchi, interaksiya yo'q.
2. Overfitting yo'q: train va test xatolari bir xil.
3. Xatoning katta qismi kamaytirib bo'lmaydigan shovqin: mijozlar soni aniq ma'lum bo'lsa ham, guruch xatosi ~4.5% qoladi.
4. Kam uchraydigan hodisalar (Hayit, bayram) eng kam ishonchli. Ma'lumot to'plangan sari model yaxshilanadi.
5. Model kalendar, tadbirlar rejasi va ob-havo prognozi asosida **1–7 kun oldinga** bashorat beradi, natija **P10/P50/P90** oralig'i bilan chiqadi.

**Cheklovlar.**
- Ob-havo prognoz emas, haqiqiy kuzatuv: real hayotdagi xato biroz yuqoriroq bo'ladi.
- Test faqat qish va bahor oylarini qamraydi (2025-yanvar–iyun). Yoz va kuz oylaridagi aniqlik faqat train ichidagi CV bilan baholangan.
- 2025-yilda Shanba kunlari talabi pasaygani kuzatildi. Modelni muntazam (masalan, oyda bir marta) yangi ma'lumot bilan qayta o'qitish kerak.
- Narxlar o'zgarishi va yangi raqobatchilar kabi omillar ma'lumotda yo'q.